In [4]:
import pandas as pd
import numpy as np
import win32com.client as com
import os

In [2]:
FOLDER = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
INSUMOS_IMEPLAN = r"Insumos IMEPLAN\Transporte Publico\FRECUENCIAS"
RED_FOLDER = r"Red Base GDL\RedBase Final\version completa"

## Agregar frecuencias de Line Routes a Time Profiles

In [3]:
#Red base GDL (con 2,203 zonas)
red_base = os.path.join(FOLDER, RED_FOLDER, "RedFinal - completa.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [15]:
# Read Time Profiles from Visum
time_profiles = pd.DataFrame({
    "LineName": [i[1] for i in Visum.Net.TimeProfiles.GetMultiAttValues("LineName")],
    "LineRouteName": [i[1] for i in Visum.Net.TimeProfiles.GetMultiAttValues("LineRouteName")],
})

# Read frecuencias from excel IMEPLAN
frequencies = pd.read_excel(os.path.join(FOLDER, INSUMOS_IMEPLAN, "Cruce_rutas_Visum_frecuencias_PARA_ENTREGAR.xlsx"), sheet_name="Resultado")
frequencies = frequencies[["Nombre_Line_Visum", "Nombre_LineRoute_Visum", "FRECUENCIA_HMD_FINAL"]]
frequencies = frequencies.rename(columns={"Nombre_LineRoute_Visum":"LineRouteName"})
# frecuencia de min a seg
frequencies["FRECUENCIA_HMD_FINAL"] = frequencies["FRECUENCIA_HMD_FINAL"]*60

# Merge on LineRoute Name
time_profiles = time_profiles.merge(
    frequencies[["LineRouteName", "FRECUENCIA_HMD_FINAL"]],
    on="LineRouteName",
    how="left"
)
time_profiles.head()

,LineName,LineRouteName,FRECUENCIA_HMD_FINAL
0,C01,C01_2,390.0
1,C01,C01_1,390.0
2,C03,C03_1,810.0
3,C08,C08_1,579.0
4,C08,C08_2,579.0


In [16]:
time_profiles_visum = Visum.Net.TimeProfiles

time_profiles = time_profiles.reset_index(drop=True)
time_profiles.index = time_profiles.index + 1

freq_values = list(zip(time_profiles.index, time_profiles["FRECUENCIA_HMD_FINAL"].astype(float)))

time_profiles_visum.SetMultiAttValues("FRECUENCIA", freq_values)